# S8c 검사 모델: Sonnet 에이전트 파일럿 재검사

검사를 luna 스크립트 호출에서 Sonnet 검사 에이전트(`multidoc-checker`, 검사 지시 `check_v3`)로 바꾼 뒤 파일럿 40개를 다시 판정한 결과를 luna 기록·사람 검수와 비교하는 근거 자료다(`docs/slices/08c-checker-agent.md`). 이 노트북을 보고 D-14에 재검사 결과와 사실 뒷받침 판정 채택 여부를 기록한다.

- luna 기록은 `data/multidoc/pilot_v4_luna/check/`, Sonnet 기록은 `data/multidoc/check/`에 있다. 두 기록 모두 지금 코드의 판정 규칙으로 다시 판정해 비교한다
- 사람 검수는 luna·Sonnet(진단용 비교, `check_v2`) 판정이 갈렸던 5건이다(`data/multidoc/review/g3_pilot_split5_review.md`)

In [1]:
import hashlib
import json
import os
import re
import time
import urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from rag.index import load_cfg
from rag.multidoc.check import judge, norm, quote_probs
from rag.multidoc.prepare import load_docs

In [2]:
ROOT = Path.cwd().parent
cfg = load_cfg()
MD = ROOT / "data/multidoc"
SONNET = ROOT / cfg.paths.multidoc_check
LUNA = MD / "pilot_v4_luna/check"
REVIEW = MD / "review/g3_pilot_split5_review.md"
CALLS = (
    MD / "quote_semantic/calls.jsonl"
)  # 08_02와 같은 Jev 호출 기록(같은 요청은 다시 부르지 않는다)
SEED = 20261007
N_SUPPORT = 30  # 뒷받침 양성·음성 묶음 크기
# 08_02와 같은 질문. 바꾸면 두 실험을 비교할 수 없다
SUPPORT = {
    "type": "noul",
    "instructions": "인용이 사실을 뒷받침하는가?",
    "criteria": {
        "true": "인용의 내용으로 사실의 핵심(누가, 무엇을, 어떤 수치나 입장)이 확인된다.",
        "false": ("인용이 사실과 다른 내용을 말하거나, 인용으로 사실의 핵심을 확인할 수 없다."),
    },
}
load_dotenv(ROOT / ".env")

True

In [3]:
pools = {
    p["pool_id"]: p
    for p in map(json.loads, (ROOT / cfg.paths.multidoc_pools).open(encoding="utf-8"))
}
s = cfg.multidoc.check.quote_semantic


def load(d):
    return {f.stem: json.loads(f.read_text(encoding="utf-8")) for f in sorted(d.glob("*.json"))}


def rejudge(r):
    # 저장된 응답을 지금 판정 규칙으로 다시 판정한다(check 재판정과 같은 인자)
    if r["gen"]["status"] != "ok":
        return r["verdict"]
    return judge(
        r["gen"],
        r["elements"],
        json.loads(r["response"])["answerable"],
        r["pool_doc_ids"],
        contexts,
        cfg.multidoc.gen.max_gold,
        pools[r["pool_id"]]["seed_doc_ids"],
        probs=quote_probs(r, s),
        min_prob=s.min_prob,
    )

In [4]:
sonnet, luna = load(SONNET), load(LUNA)
docs = load_docs(cfg, [pools[k] for k in sonnet])
contexts = {d: r["context"] for d, r in docs.items()}
print(len(sonnet), len(luna), sorted(sonnet) == sorted(luna), len(contexts))

40 40 True 380


## 1. 판정 비교

In [5]:
rows = []
for k, r in sonnet.items():
    vs, vl = rejudge(r), rejudge(luna[k])
    rows.append(
        {
            "pool_id": k,
            "type": r["type"],
            "gen": r["gen"]["status"],
            "luna": vl["passed"],
            "sonnet": vs["passed"],
            "luna_reasons": ",".join(vl["reasons"]),
            "sonnet_reasons": ",".join(vs["reasons"]),
            "luna_gold": len(vl["gold_doc_ids"]),
            "sonnet_gold": len(vs["gold_doc_ids"]),
            "gen_gold": len(r["gen"].get("seed_doc_ids") or []),
            "gold_same": set(vl["gold_doc_ids"]) == set(vs["gold_doc_ids"]),
        }
    )
cmp = pd.DataFrame(rows)
ok = cmp[cmp["gen"] == "ok"]
print(cmp.shape, "생성 ok", len(ok))
pd.crosstab(
    ok["luna"], ok["sonnet"], rownames=["luna 통과"], colnames=["Sonnet 통과"], margins=True
)

(40, 11) 생성 ok 31


Sonnet 통과,False,True,All
luna 통과,,,
False,11,5,16
True,0,15,15
All,11,20,31


In [6]:
# 기록에 저장된 판정과 재판정이 같은지(Sonnet 기록은 방금 판정했으므로 같아야 한다)
print(
    "Sonnet 저장 판정과 다름",
    sum(rejudge(r) != r["verdict"] for r in sonnet.values()),
    "| luna 저장 판정과 다름",
    sum(rejudge(r) != r["verdict"] for r in luna.values()),
)

Sonnet 저장 판정과 다름 0 | luna 저장 판정과 다름 0


In [7]:
# 불통과 사유별 건수(생성 ok 31개)
pd.DataFrame(
    {
        "luna": Counter(x for v in ok["luna_reasons"] for x in v.split(",") if x),
        "Sonnet": Counter(x for v in ok["sonnet_reasons"] for x in v.split(",") if x),
    }
).fillna(0).astype(int)

,luna,Sonnet
seed_mismatch,10,10
substitutable,12,7
unanswerable,1,0


In [8]:
# 판정이 갈린 후보 집합
ok[ok["luna"] != ok["sonnet"]][
    [
        "pool_id",
        "luna",
        "sonnet",
        "luna_reasons",
        "sonnet_reasons",
        "gen_gold",
        "luna_gold",
        "sonnet_gold",
    ]
]

,pool_id,luna,sonnet,luna_reasons,sonnet_reasons,gen_gold,luna_gold,sonnet_gold
10,conf-0010,False,True,"substitutable,seed_mismatch",,3,4,3
11,conf-0011,False,True,unanswerable,,3,3,3
18,conf-0018,False,True,substitutable,,4,4,4
29,law-0009,False,True,substitutable,,3,3,3
35,questioner-0005,False,True,substitutable,,5,5,5


In [9]:
# 두 검사가 찾은 정답 문서 집합이 같은 비율(생성 ok)
print(f"정답 문서 집합 일치 {ok['gold_same'].sum()}/{len(ok)}")
ok[~ok["gold_same"]][["pool_id", "gen_gold", "luna_gold", "sonnet_gold", "luna", "sonnet"]]

정답 문서 집합 일치 25/31


,pool_id,gen_gold,luna_gold,sonnet_gold,luna,sonnet
3,conf-0003,3,2,3,False,False
8,conf-0008,5,5,6,False,False
10,conf-0010,3,4,3,False,True
13,conf-0013,3,6,5,False,False
16,conf-0016,4,7,6,False,False
27,law-0007,4,8,8,False,False


## 2. G3 갈린 5건과 사람 검수

In [10]:
text = REVIEW.read_text(encoding="utf-8")
human = dict(re.findall(r"^## (\S+)\n.*?^- 맞는 판정: (\S+)", text, flags=re.S | re.M))
# 사람 판정 'Sonnet'은 통과가 맞음, 'luna'는 불통과가 맞음
g3 = cmp[cmp["pool_id"].isin(human)].assign(
    사람_통과=lambda d: d["pool_id"].map(lambda k: human[k] == "Sonnet")
)
g3[["pool_id", "사람_통과", "luna", "sonnet", "luna_reasons", "sonnet_reasons"]]

,pool_id,사람_통과,luna,sonnet,luna_reasons,sonnet_reasons
5,conf-0005,True,False,False,"substitutable,seed_mismatch",seed_mismatch
11,conf-0011,False,False,True,unanswerable,
14,conf-0014,False,False,False,seed_mismatch,"substitutable,seed_mismatch"
18,conf-0018,True,False,True,substitutable,
35,questioner-0005,True,False,True,substitutable,


In [11]:
print(
    "사람과 같은 판정 | luna",
    int((g3["luna"] == g3["사람_통과"]).sum()),
    "| Sonnet(check_v3)",
    int((g3["sonnet"] == g3["사람_통과"]).sum()),
    "/",
    len(g3),
)

사람과 같은 판정 | luna 2 | Sonnet(check_v3) 3 / 5


- 사람과 같은 판정은 Sonnet(`check_v3`) 3건(conf-0014·0018, questioner-0005), luna 2건(conf-0011·0014)이다
- conf-0005: 사람은 정답 문서 D1·D5로 통과가 맞다고 봤는데, 이번 Sonnet은 D2(근로시간면제자의 휴직·공가 문제)를 근거로 더해 seed_mismatch가 됐다. 진단용 비교(`check_v2`)에서는 통과였다
- conf-0011: 사람은 기대 답에 원문에 없는 인과가 있어 unanswerable이 맞다고 봤다. Sonnet은 두 번 모두 answerable로 판정했다. 생성 쪽 기대 답의 결함은 검사가 질의와 원문만 보므로 잡지 못한다

## 3. 검사 출력의 모양

In [12]:
def shape(recs, name):
    els = [el for r in recs.values() if r["gen"]["status"] == "ok" for el in r["elements"]]
    sup = [x for el in els for x in el["support"]]
    exact = [
        x["doc_id"] in contexts and norm(x["quote"]) in norm(contexts[x["doc_id"]]) for x in sup
    ]
    return {
        "검사": name,
        "요소": len(els),
        "인용": len(sup),
        "요소당 인용": round(len(sup) / len(els), 2),
        "원문에 그대로 있는 인용": f"{sum(exact)}/{len(exact)}",
        "Jev 판정 기록": sum(len(r.get("quote_checks", [])) for r in recs.values()),
    }


pd.DataFrame([shape(luna, "luna"), shape(sonnet, "Sonnet")])

,검사,요소,인용,요소당 인용,원문에 그대로 있는 인용,Jev 판정 기록
0,luna,154,413,2.68,399/413,14
1,Sonnet,177,205,1.16,205/205,0


## 4. 수율

In [13]:
y = cmp.groupby("type").agg(
    후보집합=("pool_id", "size"),
    생성ok=("gen", lambda g: (g == "ok").sum()),
    luna통과=("luna", "sum"),
    Sonnet통과=("sonnet", "sum"),
)
y.loc["합계"] = y.sum()
y["Sonnet 수율"] = (y["Sonnet통과"] / y["후보집합"]).round(3)
y["luna 수율"] = (y["luna통과"] / y["후보집합"]).round(3)
y

,후보집합,생성ok,luna통과,Sonnet통과,Sonnet 수율,luna 수율
type,,,,,,
conf,20,16,5,8,0.4,0.250
law,10,10,6,7,0.7,0.600
questioner,10,5,4,5,0.5,0.400
합계,40,31,15,20,0.5,0.375


In [14]:
# 후보 집합당 수율을 그대로 늘린 본 생성 예상
# (유형별 준비 수 n_pools_per_type, 파일럿 포함 전체 다시 생성)
n_pools = pd.Series(dict(cfg.multidoc.n_pools_per_type))
y0 = y.drop("합계")
est = pd.DataFrame(
    {
        "준비 수": n_pools,
        "Sonnet 예상": (y0["Sonnet 수율"] * n_pools).round(),
        "luna 예상": (y0["luna 수율"] * n_pools).round(),
    }
)
est.loc["합계"] = est.sum()
est.astype(int)

,준비 수,Sonnet 예상,luna 예상
conf,350,140,88
law,250,175,150
questioner,350,175,140
합계,950,490,378


## 5. 사실 뒷받침 재실험

08_02에서는 검사 출력이 질의와 답변처럼 여러 내용을 한 사실에 합쳐 써서, 인용이 사실의 일부만 담는 양성이 많았고 뒷받침 판정을 채택하지 않았다. `check_v3`은 "사실 하나에 한 가지 내용"을 지시한다. 같은 질문(`SUPPORT`)과 같은 묶음 방식으로 Sonnet 검사 기록에서 다시 잰다.

- 양성: Sonnet 검사의 (사실, 그 사실의 인용) 30쌍
- 음성: 같은 후보 집합의 다른 요소에서 온 인용 30쌍. 그 인용의 문서가 원래 요소의 근거 문서가 아닐 때만

In [45]:
cache = {r["key"]: r for r in map(json.loads, CALLS.read_text(encoding="utf-8").splitlines())}


def jev(state, question):
    body = {"model": s.model, "state": state, "questions": {"q": question}}
    key = hashlib.sha256(json.dumps(body, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    if key not in cache:
        req = urllib.request.Request(
            s.url,
            data=json.dumps(body).encode(),
            headers={
                "Authorization": "Bearer " + os.environ[cfg.multidoc.check.api_key_env],
                "Content-Type": "application/json",
            },
        )
        t = time.time()
        with urllib.request.urlopen(req, timeout=60) as r:
            resp = json.loads(r.read().decode())
        rec = {"key": key, "tag": "08_03", "latency": round(time.time() - t, 3), "response": resp}
        with CALLS.open("a", encoding="utf-8", newline="\n") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        cache[key] = rec
    rec = cache[key]
    return {"p": rec["response"]["answers"]["q"]["noul"], "cost": rec["response"]["usage"]["cost"]}

In [46]:
pos, neg = [], []
for r in sonnet.values():
    els = r.get("elements") or []
    for i, el in enumerate(els):
        pos += [(r["pool_id"], i, el["fact"], x["quote"]) for x in el["support"] if x["doc_id"]]
        docs_i = {x["doc_id"] for x in el["support"]}
        neg += [
            (r["pool_id"], i, el["fact"], x["quote"])
            for j, other in enumerate(els)
            if j != i
            for x in other["support"]
            if x["doc_id"] and x["doc_id"] not in docs_i
        ]
cols = ["pool_id", "el", "fact", "quote"]
pos, neg = pd.DataFrame(pos, columns=cols), pd.DataFrame(neg, columns=cols)
print("후보 양성", len(pos), "음성", len(neg))
pairs = pd.concat(
    [
        pos.sample(N_SUPPORT, random_state=SEED).assign(group="양성"),
        neg.sample(N_SUPPORT, random_state=SEED).assign(group="음성"),
    ],
    ignore_index=True,
)
pairs.groupby("group")["pool_id"].nunique()

후보 양성 205 음성 808


group
양성    19
음성    17
Name: pool_id, dtype: int64

In [47]:
n_before = len(cache)
res = pd.DataFrame(
    [
        {**x._asdict(), **jev({"사실": x.fact, "인용": norm(x.quote)}, SUPPORT)}
        for x in pairs.itertuples()
    ]
)
print(len(res), "건, 새 호출", len(cache) - n_before, f"비용 ${res['cost'].sum():.5f}")
res.groupby("group")["p"].describe()[["count", "min", "25%", "50%", "75%", "max"]].round(3)

60 건, 새 호출 60 비용 $0.00116


,count,min,25%,50%,75%,max
group,,,,,,
양성,30.0,0.19,0.678,0.795,0.852,0.93
음성,30.0,0.02,0.053,0.095,0.182,0.59


In [48]:
pd.DataFrame(
    [
        {
            "min_prob": t,
            "양성 인정": int((res[res["group"] == "양성"]["p"] >= t).sum()),
            "음성 인정": int((res[res["group"] == "음성"]["p"] >= t).sum()),
        }
        for t in [0.3, 0.5, 0.7, 0.8, 0.9]
    ]
)

,min_prob,양성 인정,음성 인정
0,0.3,29,3
1,0.5,26,1
2,0.7,22,0
3,0.8,15,0
4,0.9,5,0


In [49]:
# 라벨 확인 대상: 확률이 낮은 양성과 높은 음성. 사실과 인용은 앞부분만
chk = pd.concat(
    [res[res["group"] == "양성"].nsmallest(6, "p"), res[res["group"] == "음성"].nlargest(6, "p")]
)
pd.set_option("display.max_colwidth", 70)
chk.assign(fact=chk["fact"].str[:60], quote=chk["quote"].map(lambda q: norm(q)[:60]))[
    ["group", "pool_id", "el", "p", "fact", "quote"]
]

,group,pool_id,el,p,fact,quote
21,양성,law-0007,6,0.19,2021년 11월 25일 소위에서 성과관리 제도가 국무조정실 업무평가와 중복되는지가 논의되었다,"그런데 앞으로 최소화하는 방향으로 개선해 나가겠다, 그러니까 아직은 개선이 안 된 거잖아요."
15,양성,questioner-0006,1,0.43,SK지오센트릭과 S-OIL 중대재해 사고 수사에서 경영책임자와 사업주를 수사하지 않는 점을 질의함,그렇다면 경영책임자나 이런 사업주를 불러서 수사를 해야 되지 않겠습니까?
18,양성,conf-0008,3,0.44,기존 민간어린이집이 입주자대표회의 동의와 지자체 신청으로 국공립으로 전환하면 리모델링비(평균 1억)를 지원한,지금 기존에 있던 민간어린이집 같은 경우에는 국공립어린이집 신축사업 예산으로 리모델링 비용을 지원하고 아니면
19,양성,questioner-0003,3,0.47,헤이트 스피치의 강도와 앞으로의 전망을 물었다,헤이트 스피치 문제 한 3분의 1로 줄어들었습디다. 그 강도가 어떻습니까?
23,양성,questioner-0006,2,0.56,울산청 경찰수사심의위원회가 1년에 네 번만 개최되어 전국에서 가장 적은 이유를 질의함,"그런데 제가 울산청을 이렇게 보니까 1년에 두 차례, 총 네 번 회의를 개최했더라고요."
3,양성,conf-0019,5,0.63,필요성: 전방 대대급 망은 전송속도가 낮아 군 간 접속과 동영상 교육이 불가능하다,지금 전방의 실정을 보면 대대급에 2MG 정도의 전송속도를 가진 망이 형성되어 있습니다.
55,음성,conf-0016,7,0.59,정부 입장: 민자 적격성 검토와 민간투자심의위원회 심의 등 운영 과정의 스크린으로 우려를 관리할 수 있고 문,공공의 목적에 부합할 수 있는 심의나 점검장치를 갖는다든가 기타 또……
38,음성,questioner-0005,3,0.49,참전유공자 약제비 감면에 상응하는 내년 예산 반영을 요구했다,그런데 정작 국회 예산에는 그것을 대체해 줄 그런 예산이 전혀 마련돼 있지가 않습니다.
53,음성,law-0008,3,0.42,3월 4일 소위에서 세무사에게 검증을 맡기지 않고 세무서가 직접 수행하면 되지 않느냐는 쟁점이 제기되었다,그래서 이것은 조사권을 민간에 위탁했다 이런 것하고는 상당히 내용이 다르다고 보고 있습니다.
34,음성,conf-0006,3,0.23,"이언주 소위원장은 총리 주재로 하되 부위원장을 산자부장관, 위원들을 차관급으로 두는 절충안을 물었다","단지 저희가 걱정되는 부분은 연관 법들이 전체적으로 부처에서, 여러 상임위에서 같이 논의가 되는데 위원장을"


메인이 원문 인용과 요소 전체를 보고 확인한 라벨(2026-10-07). 확률이 가장 낮은 양성 6쌍은 모두 인용이 사실의 일부만 담는다.

- 사실이 인용보다 넓음 4: questioner-0006 el1(사업장 이름), questioner-0006 el2("전국에서 가장 적은 이유"), questioner-0003 el3("전망"), conf-0019 el5("불가능하다"). 사실 문장에 인용에 없는 내용을 더 써 넣었다
- 요소의 근거 여러 개 중 하나 2: law-0007 el6(0.19, 중복 쟁점은 같은 요소의 다른 인용에 있다), conf-0008 el3(입주자대표회의·1억은 같은 요소의 다른 인용에 있다)
- 음성 상위 3쌍(0.42\~0.59)은 모두 실제로 뒷받침하지 않는다. 같은 회의의 비슷한 주제라 확률이 올랐다(conf-0016 el7은 위원 제안을 정부 입장의 근거로 짝지은 쌍)

08_02(luna 기록, 0.5에서 양성 24/30·음성 2/30)보다 분리가 조금 나아졌다(26/30·1/30). 다만 0.5 아래로 떨어지는 양성은 모두 부분 뒷받침이라, 인용 단위로 근거를 빼는 규칙을 두면 같은 요소의 다른 인용이 사실을 받치는 경우에도 문서가 빠진다. 이번 재검사에서 사람과 어긋난 conf-0005(무관 문서 추가)와 conf-0011(원문에 없는 인과)은 인용이 사실을 뒷받침하는지의 문제가 아니어서 이 판정으로 잡히지 않는다.